In [0]:
from abc import ABC, abstractmethod

import pandas as pd
import yaml

In [0]:
class InterfaceDatabaseFecher(ABC):
    
    @abstractmethod
    def get_catalog_id(self, catalog_name):
        pass

    @abstractmethod
    def get_schema_id(self, catalog_id, schema_name):
        pass

class DatabaseFetcher():

    def __init__(self):
        pass

    def get_catalog_id(self, catalog_name):

        sql = f"""SELECT catalog_id FROM governance_prod.metadata.catalogs WHERE catalog_name = '{catalog_name}'"""
        result = spark.sql(sql).collect()
        
        if len(result) != 1:
            raise Exception('Unique catalog not found')

        return result[0]['catalog_id']

    def get_schema_id(self, catalog_id, schema_name):

        sql = f"""SELECT schema_id FROM governance_prod.metadata.schemas 
                  WHERE catalog_id = {catalog_id} AND schema_name = '{schema_name}'"""
        result = spark.sql(sql).collect()
        
        if len(result) != 1:
            raise Exception('Unique schema not found')

        return result[0]['schema_id']

In [0]:
class AbstractSurrogateKeyManager(ABC):

    def __init__(self, catalog_name: str, schema_name: str, table_name: str, base_values: dict, surrogate_column: str):
        self.complete_table_name = f'{catalog_name}.{schema_name}.{table_name}'
        self.base_values = base_values
        self.surrogate_column = surrogate_column

        self.where_string = None
        self.sql_string = None
        self.surrogate_key = None

    def get_surrogate_key(self):
        
        self.parse_base_values()
        self.generate_where_string()
        self.generate_sql_string()
        self.get_identifier()

        return self.surrogate_key

    def parse_base_values(self):

        for key, value in self.base_values.items():
            
            if pd.isnull(value):
                raise Exception('Base value cannot be null')

            str_value = str(value).strip()
            if str_value == '':
                raise Exception('Base value cannot be empty')

            self.base_values[key] = str_value

    def generate_where_string(self):

        where_string = []
        for key, value in self.base_values.items():
            where_string.append(f"CAST({key} AS STRING)='{value}'")
        
        self.where_string = ' AND '.join(where_string)

    @abstractmethod
    def generate_sql_string(self):
        pass

    def get_identifier(self):
      
        table_record = spark.sql(self.sql_string).collect()
            
        if len(table_record) > 1:
            raise Exception('Unicity violation on governance metadata for tables')

        elif len(table_record) == 1:
            self.surrogate_key = table_record[0]['identifier']

        else:
            max_identifier = spark.sql(f"""SELECT MAX({self.surrogate_column}) AS identifier 
                                           FROM {self.complete_table_name}""").collect()[0]['identifier']
            
            if pd.isnull(max_identifier):
                self.surrogate_key = 1

            else:
                self.surrogate_key = max_identifier + 1

class SurrogateKeyManager(AbstractSurrogateKeyManager):

    def generate_sql_string(self):
        self.sql_string = f"""SELECT {self.surrogate_column} AS identifier 
                                         FROM {self.complete_table_name} 
                                         WHERE {self.where_string}"""

class SurrogateKeyManagerMetaTable(AbstractSurrogateKeyManager):

    def generate_sql_string(self):
        self.sql_string = f"""SELECT t.{self.surrogate_column} AS identifier 
                              FROM {self.complete_table_name} t
                              INNER JOIN governance_prod.metadata.schemas s ON t.schema_id = s.schema_id
                              INNER JOIN governance_prod.metadata.catalogs c ON s.catalog_id = c.catalog_id
                              WHERE {self.where_string}"""

class SurrogateKeyFactory():

    def __init__(self, catalog_name: str, schema_name: str, table_name: str, base_values: dict, surrogate_column: str):

        self.surrogate_key_manager = None

        if catalog_name == 'governance_prod' and schema_name == 'metadata' and table_name == 'tables':
            self.surrogate_key_manager = SurrogateKeyManagerMetaTable(catalog_name=catalog_name, 
                                                                 schema_name=schema_name, 
                                                                 table_name=table_name,
                                                                 base_values=base_values, 
                                                                 surrogate_column=surrogate_column)
        else:
            self.surrogate_key_manager = SurrogateKeyManager(catalog_name=catalog_name, 
                                                             schema_name=schema_name, 
                                                             table_name=table_name, 
                                                             base_values=base_values, 
                                                             surrogate_column=surrogate_column)

    def get_surrogate_key(self):
        return self.surrogate_key_manager.get_surrogate_key()


In [0]:
path_base = '/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/governance/metadata/assets/'
domain = 'medallion'
environment = 'dev'
source = 'healthsys'
plain_table = 'cat_cie_10'

path = f'{path_base}/{domain}/{source}/{plain_table}.yml'

with open(path) as file:
    try:
        metadata = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [0]:
print(metadata)

In [0]:
for layer in metadata['layers']:
    layer_name = layer['layer']
    catalog_name = f'{domain}_{environment}'
    table_name = layer_name +  '_' + metadata['name']
    
    etl_module = None
    schema_name = None
    quality = None
    table_type = None

    if layer_name == 'raw':
        etl_module = metadata['name'] + '__load'
        schema_name = f'bronze_{source}'
        quality = 'bronze'
        table_type = 'table'

    database_fetcher = DatabaseFetcher()
    catalog_id = database_fetcher.get_catalog_id(catalog_name)
    schema_id = database_fetcher.get_schema_id(catalog_id, schema_name)
    
    base_dict = {'catalog_name': catalog_name, 'schema_name': schema_name, 'table_name': table_name, 'version': 1}
    surrogate_key_factory = SurrogateKeyFactory(catalog_name='governance_prod', schema_name='metadata', 
                                                table_name='tables', base_values=base_dict, surrogate_column='table_id')
    

    table_id = surrogate_key_factory.get_surrogate_key()

    write_mode = layer['write_mode']
    version = layer['version']
    current_flag = True
    valid_from = '2025-01-01 00:00:00'
    valid_to = '2200-01-01 00:00:00'
    description = layer['description']
    owner = layer['owner']
    retention_policy = layer['retention_policy']

    print(table_id)
    print(schema_id)
    print(table_name)
    print(etl_module)
    print(write_mode)   
    print(quality)
    print(table_type)
    print(version)
    print(current_flag)
    print(valid_from)
    print(valid_to)
    print(description)
    print(owner)
    print(retention_policy)

    array_data_columns = []
    i = 1
    for column in layer['schema']:

        column_name = column['column']
        base_dict = {'table_id': table_id, 'column_name': column_name}
        surrogate_key_factory = SurrogateKeyFactory(catalog_name='governance_prod', schema_name='metadata', 
                                                    table_name='tables_detail', base_values=base_dict, 
                                                    surrogate_column='column_id')
    
        column_id = surrogate_key_factory.get_surrogate_key()

        data_column = {}
        data_column['table_id'] = table_id
        data_column['column_id'] = column_id
        data_column['column_name'] = column_name
        data_column['data_type'] = column['data_type']
        data_column['ordinal_position'] = i
        data_column['is_nullable'] = column['is_nullable']
        data_column['is_partition'] = column['is_partition']
        data_column['comment'] = column['comment']
        array_data_columns.append(data_column)
        i = i + 1

    for element in array_data_columns:
        print(element)
    

    

In [0]:
array_table_id = [1]
array_schema_id = [3]
array_name = ['raw_cat_cie_10']
array_etl_module = ['cat_cie_10__load']
array_write_mode = ['overwrite_partition']
array_quality = ['bronze']
array_table_type = ['table']
#array_storage_location = ['Lakehouse']
#array_input_format = ['csv']
array_version = [1]
array_current_flag = [True]
array_valid_from = ['2025-11-19 00:00:00']
array_valid_to = ['2200-01-01 00:00:00']
array_description = ['Catalog of CIE codes as in the original source in delta format']
array_owner = ['armando.n90@gmail.com']
array_retention_policy = ['permanent']

columns_comments = {
    "table_id": "Identifier of the table",
    "schema_id": "Identifier of the schema",
    "table_name": "Name of the table",
    "etl_module": "Name of the ETL module that created the table",
    "write_mode": "Write mode of the table among overwrite_partition, overwrite, merge and cdc",
    "quality": "Quality of the table among bronze, silver and gold",
    "table_type": "Type of the table among table, view and materialized",
    #"storage_location": "Storage location of the table if external",
    #"input_format": "Input format of the table among csv, json and yaml",
    "version": "Version of the table",
    "current_flag": "Flag to indicate if the table is current",
    "valid_from": "Date of validity of the table",
    "valid_to": "Date of expiration of the table",
    "description": "Description of the table",
    "owner": "Owner of the table",
    "retention_policy": "Retention policy of the table, currently permanent",
}

metadata_table = spark.createDataFrame(data = list(zip(array_table_id, array_schema_id, array_name, array_etl_module, array_write_mode, array_quality, array_table_type, array_version, array_current_flag, array_valid_from, array_valid_to, array_description, array_owner, array_retention_policy)), schema=['table_id', 'schema_id', 'table_name', 'etl_module', 'write_mode', 'quality', 'table_type', 'version', 'current_flag', 'valid_from', 'valid_to', 'description', 'owner', 'retention_policy'])

print(metadata_table.count())
metadata_table.show(3)

metadata_table.write.format('delta').mode('overwrite').saveAsTable('governance_prod.metadata.tables')

for column, comment in columns_comments.items():
    spark.sql(f"ALTER TABLE governance_prod.metadata.tables ALTER COLUMN {column} COMMENT '{comment}'")

In [0]:
array_table_id = [1] * 2
array_column_id = [1, 2] 
array_column_name = ['cie_code', 'cie_name']
array_data_type = ['string', 'string']
array_ordinal_position = [1, 2]
array_is_nullable = [False, False]
array_is_partition = [False, False]
array_comment = ['CIE Code', 'CIE Name'] #, 'Identifier of the ingested batch', 'Date of loading', 'Name of the ETL module']

columns_comments = {
    "table_id": "Identifier of the table",
    "column_id": "Identifier of the column",
    "column_name": "Name of the column",
    "data_type": "Data type of the column",
    "ordinal_position": "Ordinal position of the column",
    "is_nullable": "Flag to indicate if the column is nullable",
    "is_partition": "Flag to indicate if the column is a partition",
    "comment": "Comment of the column"
}

metadata_table = spark.createDataFrame(data = list(zip(array_table_id, array_column_id, array_column_name, array_data_type, array_ordinal_position, array_is_nullable, array_is_partition, array_comment)), schema=['table_id', 'column_id', 'column_name', 'data_type', 'ordinal_position', 'is_nullable', 'is_partition', 'comment'])

print(metadata_table.count())
metadata_table.show(3)

metadata_table.write.format('delta').mode('overwrite').saveAsTable('governance_prod.metadata.tables_detail')

for column, comment in columns_comments.items():
    spark.sql(f"ALTER TABLE governance_prod.metadata.tables_detail ALTER COLUMN {column} COMMENT '{comment}'")